# Data Exploration — Amazon ESCI Product Sample
Explore the 500-row product sample, entity type distributions, and BIO label examples.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import json
from collections import Counter
from pathlib import Path

df = pd.read_csv('../data/samples/products_sample.csv')
print(f'Sample shape: {df.shape}')
df.head(3)

In [ ]:
# Title length distribution
title_col = 'title' if 'title' in df.columns else 'product_title'
df['title_len'] = df[title_col].astype(str).str.split().str.len()
fig, ax = plt.subplots(figsize=(10, 4))
df['title_len'].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Title length (words)')
ax.set_ylabel('Count')
ax.set_title('Product Title Length Distribution')
plt.tight_layout()
plt.show()
print(df['title_len'].describe())

In [ ]:
# Top-20 brands
brand_col = 'brand' if 'brand' in df.columns else None
if brand_col:
    top_brands = df[brand_col].dropna().value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 5))
    top_brands.plot(kind='barh', ax=ax, color='coral')
    ax.set_title('Top 20 Brands')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top-20 categories
cat_col = 'category' if 'category' in df.columns else ('product_type' if 'product_type' in df.columns else None)
if cat_col:
    top_cats = df[cat_col].dropna().value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 5))
    top_cats.plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Top 20 Categories')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Entity type distribution from ner_train.json
train_path = Path('../data/processed/ner_train.json')
if train_path.exists():
    label_counts = Counter()
    with open(train_path) as f:
        for line in f:
            rec = json.loads(line)
            for lbl in rec['labels']:
                if lbl != 'O':
                    entity_type = lbl.split('-', 1)[1] if '-' in lbl else lbl
                    label_counts[entity_type] += 1
    fig, ax = plt.subplots(figsize=(8, 4))
    types = list(label_counts.keys())
    counts = list(label_counts.values())
    ax.bar(types, counts, color='mediumpurple')
    ax.set_title('Entity Type Distribution (Training Set)')
    ax.set_ylabel('Token Count')
    plt.tight_layout()
    plt.show()
    print(dict(label_counts))
else:
    print('Run the data pipeline first: python -m src.data.downloader')

In [ ]:
# Show 5 annotated examples with BIO labels
if train_path.exists():
    examples = []
    with open(train_path) as f:
        for i, line in enumerate(f):
            if i >= 5: break
            examples.append(json.loads(line))
    for i, ex in enumerate(examples):
        print(f'\n--- Example {i+1} ---')
        for tok, lbl in zip(ex['tokens'], ex['labels']):
            if lbl != 'O':
                print(f'  [{lbl}] {tok}')
            else:
                print(f'  {tok}')